In [ ]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
from pathlib import Path
import numpy as np
import torch
import cv2
import os
import shutil
import sys
import csv

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

In [ ]:
detect_model = YOLO("models/11L2.pt")
pose_model = YOLO("yolo11l-pose")

In [ ]:
video_input_directory = Path("video/input")
video_output_directory = Path("video/output")   

video_input_directory.mkdir(parents=True, exist_ok=True)
video_output_directory.mkdir(parents=True, exist_ok=True)

for item in video_output_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [ ]:
video_stride = 10
ioa_thres = 0.5

detect_classes = [k for k, v in detect_model.names.items() if v != "person"]
pose_classes = [k for k, v in pose_model.names.items() if v == "person"]

for video_index, video in enumerate(video_input_directory.glob("*.mp4")):
    print(f"Video: {video}")
    cap = cv2.VideoCapture(str(video))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1

    adjusted_fps = (fps / video_stride)

    output_video = str(video_output_directory / f"video{video_index}.mp4")
    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )
    
    hoi_output_log = str(video_output_directory / f"hoi_log{video_index}.csv")
    
    frame_count = 0
    
    with open(hoi_output_log, "w") as hoi_log:
        try:
            while cap.isOpened():
                success, frame = cap.read()
                if not success:
                    break
                    
                if frame_count % video_stride == 0:
                    est_time_seconds = int(frame_count / fps)
                    minutes, seconds = divmod(est_time_seconds, 60)
                    base_log = f"{frame_count:05},{minutes:02}:{seconds:02},"

                    print(f"\rProcessing Frame: {frame_count}/{total_frames}", end="", flush=True)

                    custom_tracker = "trackers/custom_botsort.yaml"

                    detect_results = detect_model.track(
                        device=0,
                        source=frame,
                        tracker=custom_tracker,
                        persist=True,
                        conf=0.5,
                        verbose=False,
                        classes=detect_classes
                    )

                    pose_results = pose_model.track(
                        device=0,
                        source=frame,
                        tracker=custom_tracker,
                        persist=True,
                        conf=0.5,
                        verbose=False,
                        classes=pose_classes
                    )

                    d_result = detect_results[0]
                    p_result = pose_results[0]
                    
                    frame = p_result.plot(boxes=False)
                    d_boxes = d_result.boxes
                    p_boxes = p_result.boxes
                    
                    if len(d_boxes) > 0:
                        d_xyxys = d_boxes.xyxy.cpu().numpy()
                        d_clss = d_boxes.cls.cpu().numpy().astype(int)
                        d_confs = d_boxes.conf.cpu().numpy()
                        d_track_ids = d_boxes.id.cpu().numpy().astype(int) if d_boxes.id is not None else [0] * len(d_boxes)

                        for i, (x1, y1, x2, y2) in enumerate(d_xyxys):
                            d_rgb = (255, 0, 0)
                            cv2.rectangle(
                                frame,
                                (int(x1), int(y1)),
                                (int(x2), int(y2)),
                                d_rgb,
                                2
                            )
                            label = f"ID:{d_track_ids[i]} CLS:{detect_model.names[d_clss[i]]} CONF:{round(float(d_confs[i]), 2)}"
                            cv2.putText(
                                frame,
                                label,
                                (int(x1), int(y1)-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                                d_rgb,
                                2
                            )
                            
                    if len(p_boxes) > 0:
                        p_xyxys = p_boxes.xyxy.cpu().numpy()
                        p_confs = p_boxes.conf.cpu().numpy()
                        p_track_ids = p_boxes.id.cpu().numpy().astype(int) if p_boxes.id is not None else [0] * len(p_boxes)
                        kpts = p_result.keypoints.xy.cpu().numpy() if p_result.keypoints is not None else None

                        for i, (x1, y1, x2, y2) in enumerate(p_xyxys):
                            p_rgb = (0, 0, 255)
                            cv2.rectangle(
                                frame,
                                (int(x1), int(y1)),
                                (int(x2), int(y2)),
                                p_rgb,
                                2
                            )
                            label = f"ID:{p_track_ids[i]} CLS:person CONF:{round(float(p_confs[i]), 2)}"
                            cv2.putText(
                                frame,
                                label,
                                (int(x1), int(y1)-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                                p_rgb,
                                2
                            )

                    if len(d_boxes) > 0:
                        xi1 = np.maximum(p_xyxys[:, None, 0], d_xyxys[None, :, 0])
                        yi1 = np.maximum(p_xyxys[:, None, 1], d_xyxys[None, :, 1])
                        xi2 = np.minimum(p_xyxys[:, None, 2], d_xyxys[None, :, 2])
                        yi2 = np.minimum(p_xyxys[:, None, 3], d_xyxys[None, :, 3])
                        
                        inter_area = np.maximum(0, xi2 - xi1) * np.maximum(0, yi2 - yi1)
                        obj_area = (d_xyxys[:, 2] - d_xyxys[:, 0]) * (d_xyxys[:, 3] - d_xyxys[:, 1])
                        
                        ioa = np.zeros_like(inter_area)
                        valid_obj = obj_area > 0
                        ioa[:, valid_obj] = inter_area[:, valid_obj] / obj_area[valid_obj]

                        p_indices, d_indices = np.where(ioa >= ioa_thres)

                        for p_i, d_i in zip(p_indices, d_indices):

                            if kpts is not None and len(kpts) > p_i:
                                person_kpts = kpts[p_i]
                                dx1, dy1, dx2, dy2 = d_xyxys[d_i]
                                in_box = (person_kpts[..., 0] >= dx1) & (person_kpts[..., 0] <= dx2) & (person_kpts[..., 1] >= dy1) & (person_kpts[..., 1] <= dy2)
                                
                                if in_box.any():
                                    hoi_log.write(base_log + f"{p_track_ids[p_i]},{d_track_ids[d_i]},{detect_model.names[d_clss[d_i]]}\n")

                    writer.write(frame)

                frame_count += 1
                
        except Exception as e:
            if writer is not None:
                writer.release()
            print(f"\n\nCrash detected!\n{e}")
            sys.exit(1)
        finally:
            cap.release()
            writer.release()
    print(f"\nCompleted {video}\n")
print("All Done")

In [ ]:
for csv_index, csv_file in enumerate(video_output_directory.glob("hoi_log*.csv")):

    print(f"Processign Log: {csv_file}")

    interactions = {}

    with open(csv_file, 'r') as f:

        for r in csv.reader(f):
            
            # print(r)

            frm, _, p, o, n = [x.strip() for x in r]
            sec = int(frm) / 30.0
            k = (p, o)

            if k not in interactions:
                interactions[k] = {'person_id': p, 'object_id': o, 'start_time': sec - 1, 'duration': 0, 'last_sec': sec, 'object_name': n}
            elif sec > interactions[k]['last_sec']:
                interactions[k]['duration'] += sec - interactions[k]['last_sec']
                interactions[k]['last_sec'] = sec

    with open(video_output_directory / f"interaction_log_{csv_index}.txt", "w", newline="") as f, open(video_output_directory / f"interaction_log_{csv_index}.csv", "w", newline="") as f_csv:

        w = csv.writer(f)
        
        for v in interactions.values():
            if v['duration'] > 1:
                end_time = v['start_time'] + v['duration']
                f.write(f"Person ID:{v['person_id']} interacted with {v['object_name']} ID:{v['object_id']} for {v['duration']:.2f} seconds starting at {v['start_time']:.2f} seconds.\n")
                f_csv.write(f"{v['person_id']},{v['object_name']},{v['object_id']},{v['start_time']:.2f},{v['duration']:.2f},{end_time}\n")
